In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
import numpy as np
from torch.utils.data import DataLoader, random_split
from datasets.dataset import OxfordIIITPetTrainDataset, OxfordIIITPetTestDataset
from models.model_torch import train_model, save_checkpoint, load_checkpoint
import matplotlib.pyplot as plt
from torchvision.transforms.functional import to_pil_image
from PIL import Image

In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

In [ ]:
train_dataset = OxfordIIITPetTrainDataset()
test_dataset = OxfordIIITPetTestDataset()
batch_size = 64

generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset = random_split(train_dataset, [.8,.2], generator=generator)

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(len(train_dataset), len(val_dataset), len(test_dataset))

### Explore datasets

In [ ]:
for batch , (x, y) in enumerate(train_dataloader):
    print(x.shape, y.shape)

In [ ]:
image, target = train_dataset[0]
print(type(image), type(target), image.shape, target)
breed_name = train_dataset.dataset.classes[target]
print(breed_name)
# Define inverse normalization to bring pixel values back to [0, 1] range for plotting
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

# 1. Process Input Image: Convert from (C, H, W) -> (H, W, C)
img = image.permute(1, 2, 0).numpy()
img = std * img + mean  # Un-normalize
img = np.clip(img, 0, 1)  # Clip boundaries ju
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(to_pil_image(img))
plt.axis("off")
plt.show()